# 05 — Business Impact & ROI
CMS-grounded financial model for wearable-augmented VBC panel management.

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("pcp-panel-intelligence"):
        os.system("git clone https://github.com/Hannah-Hiltz/pcp-panel-intelligence.git")
    os.chdir("pcp-panel-intelligence")
    sys.path.insert(0, "src")
    os.system("pip install -q xgboost shap scikit-learn joblib")
    DATA_DIR  = "data/raw";      PROC_DIR  = "data/processed"
    FIG_DIR   = "reports/figures"; MODEL_DIR = "models"
    WEAR_DIR  = "data/raw/wearables"; PANEL_DIR = "data/raw/panel"
else:
    sys.path.insert(0, "../src")
    DATA_DIR  = "../data/raw";   PROC_DIR  = "../data/processed"
    FIG_DIR   = "../reports/figures"; MODEL_DIR = "../models"
    WEAR_DIR  = "../data/raw/wearables"; PANEL_DIR = "../data/raw/panel"

for d in [FIG_DIR, PROC_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
print(f"Environment: {'Colab' if IN_COLAB else 'Jupyter'}")


In [ ]:
df   = pd.read_csv(f"{PANEL_DIR}/patient_panel_flat.csv")
long = pd.read_csv(f"{PROC_DIR}/longitudinal_snapshots.csv")
traj = pd.read_csv(f"{PROC_DIR}/risk_trajectories.csv")
print("Data loaded.")


## Panel statistics

In [ ]:
n_hc     = df["outreach_priority"].isin(["Critical","High"]).sum()
n_wear   = df["has_wearable"].sum()
avg_gaps = df[df["outreach_priority"].isin(["Critical","High"])]["n_quality_gaps"].mean()
er_total = int(df["er_visits_12m"].sum())
hosp_tot = int(df["hospitalizations_12m"].sum())
avg_cost = df["avg_annual_cost"].mean()

print(f"Panel size:              {len(df):,}")
print(f"High/Critical:           {n_hc} ({n_hc/len(df)*100:.0f}%)")
print(f"Wearable coverage:       {n_wear} ({n_wear/len(df)*100:.0f}%)")
print(f"CMS avg annual cost/pt:  ${avg_cost:,.0f}")
print(f"Avg quality gaps (H/C):  {avg_gaps:.1f}")
print(f"ER visits (12mo):        {er_total}")
print(f"Hospitalizations (12mo): {hosp_tot}")


## ROI calculation — wearable-augmented vs. static model

In [ ]:
# Constants from CMS, AHRQ, NEJM Catalyst
AVG_HOSP_COST     = 12_000
AVG_ER_COST       = 2_200
OUTREACH_COST     = 45
DEVICE_COST_ANNUAL= 120
QUALITY_BONUS_PT  = 850_000

n_outreached = int(n_hc * 0.65)
gap_closed   = n_outreached * avg_gaps * 0.35
star_pts     = gap_closed / 50
quality_val  = star_pts * QUALITY_BONUS_PT

# Static model
hosp_avoid_static = hosp_tot * 0.18
er_avoid_static   = er_total * 0.18 * 0.6
gross_static = quality_val + hosp_avoid_static*AVG_HOSP_COST + er_avoid_static*AVG_ER_COST
cost_static  = n_outreached * OUTREACH_COST * 12
net_static   = gross_static - cost_static

# Wearable-augmented (early detection adds ~12% more hosp prevention)
hosp_avoid_wear = hosp_tot * 0.30
er_avoid_wear   = er_total * 0.25 * 0.6
gross_wear = quality_val + hosp_avoid_wear*AVG_HOSP_COST + er_avoid_wear*AVG_ER_COST
cost_wear  = n_outreached * OUTREACH_COST * 12 + n_wear * DEVICE_COST_ANNUAL
net_wear   = gross_wear - cost_wear

print(f"{'Metric':<35} {'Static model':>15} {'Wearable-augmented':>20}")
print("-"*72)
print(f"{'Hospitalizations avoided':<35} {hosp_avoid_static:>15.1f} {hosp_avoid_wear:>20.1f}")
print(f"{'ER visits avoided':<35} {er_avoid_static:>15.1f} {er_avoid_wear:>20.1f}")
print(f"{'Gross annual benefit':<35} ${gross_static:>14,.0f} ${gross_wear:>19,.0f}")
print(f"{'Program cost':<35} ${cost_static:>14,.0f} ${cost_wear:>19,.0f}")
print(f"{'Net annual benefit':<35} ${net_static:>14,.0f} ${net_wear:>19,.0f}")
print(f"{'ROI':<35} {net_static/cost_static*100:>14.0f}% {net_wear/cost_wear*100:>19.0f}%")


## Comparison waterfall

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(14,6))

for ax, label, values in [
    (axes[0], "Static model",
     [quality_val, hosp_avoid_static*AVG_HOSP_COST, er_avoid_static*AVG_ER_COST, -cost_static, net_static]),
    (axes[1], "Wearable-augmented",
     [quality_val, hosp_avoid_wear*AVG_HOSP_COST, er_avoid_wear*AVG_ER_COST, -cost_wear, net_wear]),
]:
    cats = ["Quality\nbonus","Hosp\nsavings","ER\nsavings","Program\ncost","Net\nbenefit"]
    bcolors = ["#3B6D11","#3B6D11","#3B6D11","#A32D2D","#185FA5"]
    bars = ax.bar(cats, values, color=bcolors, width=0.5, edgecolor="white")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(label, fontsize=13); ax.set_ylabel("USD (annual)")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"${x/1e3:.0f}K"))
    for bar, val in zip(bars, values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+(3000 if val>=0 else -14000),
                f"${abs(val)/1e3:.0f}K", ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/roi_comparison.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"\nWearable layer adds ${(net_wear-net_static)/1e3:.0f}K additional net value annually")


## Equity ROI — value of closing the wearable access gap

In [ ]:
# Patients in Q1-Q2 without devices who would benefit from device enrollment
no_device_low_income = df[(df["has_wearable"]==0) & (df["zip_income_quintile"]<=2)]
high_risk_no_device  = no_device_low_income[no_device_low_income["outreach_priority"].isin(["Critical","High"])]
print(f"High/Critical patients in Q1-Q2 without devices: {len(high_risk_no_device)}")
print(f"Estimated additional hospitalizations preventable with wearable: {len(high_risk_no_device)*0.25:.1f}")
print(f"Estimated additional value: ${len(high_risk_no_device)*0.25*AVG_HOSP_COST:,.0f}")
print()
print("Sociology lens: closing the device access gap is not just an equity imperative —")
print("it is also the highest-ROI investment in the program.")
print("The patients most likely to be missed are the ones who most need early detection.")
